# Prepare dataset for training and testing

This notebook loads the medical students diabetes dataset, performs minimal cleaning (drop rows missing the target and duplicates), creates a stratified test split, and saves `train_set.csv` and `test_set.csv` under the `../data/` directory.

How to use:
- Run all cells in order.
- Ensure internet access if the local file `Data/medical_students_diabetes_dataset.csv` is not present (the notebook will attempt to download a CSV export from a public Google Sheets URL).

Notes:
- This notebook aims to be reproducible: `RANDOM_STATE` controls randomness and is defined as a constant. The core logic of loading, cleaning, and splitting is preserved.


In [1]:
import os
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split

# --- Configuration constants (do not change logic) ---
DATASET_URL = 'https://docs.google.com/spreadsheets/d/1sQgs550vWZPifj0j_HJ-i0WrbXo_MDI8Vgmm0IwJCJI/export?format=csv&gid=1996190875'
DATA_DIR = Path("..") / "data"
DATASET_PATH = DATA_DIR / "medical_students_diabetes_dataset.csv"
TEST_SIZE = 0.15
RANDOM_STATE = 2005021
REQUIRED_COLUMNS = ["Diabetes"]


In [2]:
# 1. Import the dataset in a notebook environment with pandas

# Ensure data dir exists
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Load dataset: prefer local copy, otherwise download and save locally
try:
    if DATASET_PATH.exists():
        df = pd.read_csv(DATASET_PATH)
    else:
        df = pd.read_csv(DATASET_URL)
        df.to_csv(DATASET_PATH, index=False)
except Exception as e:
    raise RuntimeError(f"Failed to load dataset from local path or URL: {e}")

# Basic validation: required columns
missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in dataset: {missing_cols}")


In [3]:
# Quick exploratory checks
print("Columns:", df.columns.tolist())
print("\nTarget distribution (including NaNs):")
print(df['Diabetes'].value_counts(dropna=False))

Columns: ['Student ID', 'Age', 'Gender', 'Height', 'Weight', 'Blood Type', 'BMI', 'Temperature', 'Heart Rate', 'Blood Pressure', 'Cholesterol', 'Diabetes', 'Smoking']

Target distribution (including NaNs):
Diabetes
No     161986
NaN     20000
Yes     18014
Name: count, dtype: int64


In [4]:
# Drop rows with NaN in the target and duplicate rows
df_clean = df.dropna(subset=['Diabetes']).drop_duplicates()
print(f"Data set: {df_clean.shape[0]} samples")
print("Data set distribution after cleaning:")
print(df_clean['Diabetes'].value_counts(normalize=True))

Data set: 172451 samples
Data set distribution after cleaning:
Diabetes
No     0.899763
Yes    0.100237
Name: proportion, dtype: float64


In [5]:
# Split the full dataframe; stratify using the 'Diabetes' column
_, test_df = train_test_split(
    df_clean,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df_clean['Diabetes']
)

print(f"Test set: {test_df.shape[0]} samples")
print("Test class distribution:\n", test_df['Diabetes'].value_counts(normalize=True))

Test set: 25868 samples
Test class distribution:
 Diabetes
No     0.89976
Yes    0.10024
Name: proportion, dtype: float64


In [6]:
# Save test set as CSV
testset_path = DATA_DIR / 'test_set.csv'
os.makedirs(testset_path.parent, exist_ok=True)
test_df.to_csv(testset_path, index=False)
print(f"Test set saved to '{testset_path}'")

Test set saved to '..\data\test_set.csv'


In [7]:
# Prepare train set with all rows of data except test set
train_df = df.drop(test_df.index)
print(f"Train set: {train_df.shape[0]} samples")
print("Train class distribution:\n", train_df['Diabetes'].value_counts(normalize=True))

Train set: 174132 samples
Train class distribution:
 Diabetes
No     0.899949
Yes    0.100051
Name: proportion, dtype: float64


In [8]:
# Save train set as CSV
trainset_path = DATA_DIR / 'train_set.csv'
os.makedirs(trainset_path.parent, exist_ok=True)
train_df.to_csv(trainset_path, index=False)
print(f"Train set saved to '{trainset_path}'")

Train set saved to '..\data\train_set.csv'
